In [1]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,id,text,label
0,1,Wall St. Bears Claw Back Into the Black (Reute...,2
1,2,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,4,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,5,"Oil prices soar to all-time record, posing new...",2


In [2]:
df_train.text[0]

"Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again."

In [3]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'[\d+]', '', text)
    text = re.sub(r'\s+', ' ', text)
    
    return text

preprocess(df_train.text[0])

'wall st bears claw back into the black reuters reuters shortsellers wall streets dwindlingband of ultracynics are seeing green again'

In [4]:
df_train.text = df_train.text.apply(preprocess)
df_test.text = df_test.text.apply(preprocess)

df_train.head()

,id,text,label
0,1,wall st bears claw back into the black reuters...,2
1,2,carlyle looks toward commercial aerospace reut...,2
2,3,oil and economy cloud stocks outlook reuters r...,2
3,4,iraq halts oil exports from main southern pipe...,2
4,5,oil prices soar to alltime record posing new m...,2


In [28]:
features = 'text'
target = 'label'

X = df_train[features]
y = df_train[target]
X_final = df_test[features]

from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer(ngram_range=(1,2), max_df=0.90, min_df=2, max_features=500000, sublinear_tf=True, norm="l2", stop_words="english")
X = vect.fit_transform(X)
X_final = vect.transform(X_final)

X

<120000x388651 sparse matrix of type '<class 'numpy.float64'>'
	with 3967946 stored elements in Compressed Sparse Row format>

In [37]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

model = LinearSVC()

cross_val_score(model, X, y, cv=2, scoring='accuracy').mean()

0.903875

In [38]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [39]:
df_preds = pd.DataFrame({
    "id": df_test.id
    "label": y_preds
})

df_preds.head()

,id,label
0,120001,2
1,120002,3
2,120003,3
3,120004,3
4,120005,3


In [40]:
df_preds.to_csv("submission.csv", index=False)